In [1]:
import os
from pathlib import Path
from copy import deepcopy

import dotenv
import pandas as pd
import pyam
from pandas_openscm.io import load_timeseries_csv
import silicone.database_crunchers
import pandas_indexing

import gcages
from gcages import renaming
from gcages.cmip7_scenariomip.harmonisation import create_cmip7_scenariomip_global_harmoniser
from gcages.cmip7_scenariomip.infilling import CMIP7ScenarioMIPInfiller

dotenv.load_dotenv()

# Set up output folder
OUTPUT_FOLDER = Path(os.environ["OUTPUT_FOLDER"])
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

%load_ext autoreload
%autoreload 2

/Users/hoegner/GitHub/ar7_netzero_assessment/.venv/lib/python3.13/site-packages/jwt/api_jwt.py:147: InsecureKeyLengthWarning: The HMAC key is 20 bytes long, which is below the minimum recommended length of 32 bytes for SHA256. See RFC 7518 Section 3.2.
  return self._jws.encode(
[WARNING] 13:52:57 - pint.util: Redefining 'C' (<class 'pint.delegates.txt_defparser.plain.UnitDefinition'>)
[WARNING] 13:52:57 - pint.util: Redefining 'N' (<class 'pint.delegates.txt_defparser.plain.UnitDefinition'>)
[WARNING] 13:52:57 - pint.util: Redefining 'NOX' (<class 'pint.delegates.txt_defparser.plain.UnitDefinition'>)
[WARNING] 13:52:57 - pint.util: Redefining 'gNOX' (<class 'pint.delegates.txt_defparser.plain.UnitDefinition'>)
[WARNING] 13:52:57 - pint.util: Redefining 'tNOX' (<class 'pint.delegates.txt_defparser.plain.UnitDefinition'>)
[WARNING] 13:52:57 - pint.util: Redefining 'S' (<class 'pint.delegates.txt_defparser.plain.UnitDefinition'>)
[WARNING] 13:52:57 - pint.util: Redefining 'yr' (<class 'p

# Configuration and File Paths

In [2]:
# Input and output paths
INPUT_DATA_FILE = OUTPUT_FOLDER / "100_inputdata_20260522.csv"
INTERMEDIATE_OUTPUT = OUTPUT_FOLDER / "intermediate/101_intermediate_infilling_db.csv"
INTERMEDIATE_OUTPUT.parent.mkdir(parents=True, exist_ok=True)

# CMIP7 configuration paths
BASE_DIR = Path("../../gcages/tests/regression/cmip7-scenariomip/cmip7-scenariomip-workflow-inputs")
CMIP7_HISTORICAL_EMISSIONS = BASE_DIR / "history_cmip7_scenariomip.csv"
ANERIS_GLOBAL_OVERRIDES = BASE_DIR / "aneris-overrides-global.csv"
CMIP7_INFILLING_FILE = BASE_DIR / "infilling_db_cmip7_scenariomip.csv"
GHG_INVERSION_FILE = BASE_DIR / "cmip7_ghg_inversions.csv"

In [3]:
BASE_DIR

PosixPath('../../gcages/tests/regression/cmip7-scenariomip/cmip7-scenariomip-workflow-inputs')

In [4]:
# Database cruncher configuration for infilling missing variables
VARS_DB_CRUNCHERS = {
    "Emissions|BC": (False, silicone.database_crunchers.QuantileRollingWindows),
    "Emissions|CH4": (False, silicone.database_crunchers.QuantileRollingWindows),
    "Emissions|CO2|Biosphere": (False, silicone.database_crunchers.QuantileRollingWindows),
    "Emissions|CO2|Fossil": (False, silicone.database_crunchers.QuantileRollingWindows),
    "Emissions|CO": (False, silicone.database_crunchers.QuantileRollingWindows),
    "Emissions|N2O": (False, silicone.database_crunchers.QuantileRollingWindows),
    "Emissions|NH3": (False, silicone.database_crunchers.QuantileRollingWindows),
    "Emissions|NOx": (False, silicone.database_crunchers.QuantileRollingWindows),
    "Emissions|OC": (False, silicone.database_crunchers.QuantileRollingWindows),
    "Emissions|SOx": (False, silicone.database_crunchers.QuantileRollingWindows),
    "Emissions|NMVOC": (False, silicone.database_crunchers.QuantileRollingWindows),
    "Emissions|HFC134a": (False, silicone.database_crunchers.RMSClosest),
    "Emissions|HFC143a": (False, silicone.database_crunchers.RMSClosest),
    "Emissions|HFC227ea": (False, silicone.database_crunchers.RMSClosest),
    "Emissions|HFC23": (False, silicone.database_crunchers.RMSClosest),
    "Emissions|HFC32": (False, silicone.database_crunchers.RMSClosest),
    "Emissions|HFC4310mee": (False, silicone.database_crunchers.RMSClosest),
    "Emissions|HFC125": (False, silicone.database_crunchers.RMSClosest),
    "Emissions|SF6": (False, silicone.database_crunchers.RMSClosest),
    "Emissions|CF4": (False, silicone.database_crunchers.RMSClosest),
    "Emissions|C2F6": (False, silicone.database_crunchers.RMSClosest),
    "Emissions|C6F14": (False, silicone.database_crunchers.RMSClosest),
    "Emissions|CCl4": (True, silicone.database_crunchers.RMSClosest),
    "Emissions|CFC11": (True, silicone.database_crunchers.RMSClosest),
    "Emissions|CFC113": (True, silicone.database_crunchers.RMSClosest),
    "Emissions|CFC114": (True, silicone.database_crunchers.RMSClosest),
    "Emissions|CFC115": (True, silicone.database_crunchers.RMSClosest),
    "Emissions|CFC12": (True, silicone.database_crunchers.RMSClosest),
    "Emissions|CH2Cl2": (True, silicone.database_crunchers.RMSClosest),
    "Emissions|CH3Br": (True, silicone.database_crunchers.RMSClosest),
    "Emissions|CH3CCl3": (True, silicone.database_crunchers.RMSClosest),
    "Emissions|CH3Cl": (True, silicone.database_crunchers.RMSClosest),
    "Emissions|CHCl3": (True, silicone.database_crunchers.RMSClosest),
    "Emissions|HCFC141b": (True, silicone.database_crunchers.RMSClosest),
    "Emissions|HCFC142b": (True, silicone.database_crunchers.RMSClosest),
    "Emissions|HCFC22": (True, silicone.database_crunchers.RMSClosest),
    "Emissions|HFC152a": (True, silicone.database_crunchers.RMSClosest),
    "Emissions|HFC236fa": (True, silicone.database_crunchers.RMSClosest),
    "Emissions|HFC365mfc": (True, silicone.database_crunchers.RMSClosest),
    "Emissions|Halon1202": (True, silicone.database_crunchers.RMSClosest),
    "Emissions|Halon1211": (True, silicone.database_crunchers.RMSClosest),
    "Emissions|Halon1301": (True, silicone.database_crunchers.RMSClosest),
    "Emissions|Halon2402": (True, silicone.database_crunchers.RMSClosest),
    "Emissions|NF3": (True, silicone.database_crunchers.RMSClosest),
    "Emissions|C3F8": (True, silicone.database_crunchers.RMSClosest),
    "Emissions|C4F10": (True, silicone.database_crunchers.RMSClosest),
    "Emissions|C5F12": (True, silicone.database_crunchers.RMSClosest),
    "Emissions|C7F16": (True, silicone.database_crunchers.RMSClosest),
    "Emissions|C8F18": (True, silicone.database_crunchers.RMSClosest),
    "Emissions|cC4F8": (True, silicone.database_crunchers.RMSClosest),
    "Emissions|SO2F2": (True, silicone.database_crunchers.RMSClosest),
}

LEAD_VARIABLES = ["Emissions|CO2"]

## Step 1: Load and Prepare Input Data

In [5]:
scenario_data = load_timeseries_csv(
    INPUT_DATA_FILE,
    index_columns=["model", "scenario", "region", "variable", "unit"],
    out_columns_type=int,
    out_columns_name="year"
)

In [6]:
scenario_data_renamed = renaming.rename_variables(
    scenario_data,
    from_convention=renaming.SupportedNamingConventions.CMIP7_SCENARIOMIP,
    to_convention=renaming.SupportedNamingConventions.GCAGES
)

# Fix HFC43-10 unit naming (remove hyphen)
scenario_data_renamed = scenario_data_renamed.reset_index()
scenario_data_renamed['unit'] = scenario_data_renamed['unit'].replace(
    'kt HFC43-10/yr', 'kt HFC4310/yr'
)
scenario_data_renamed.set_index(
    ["model", "scenario", "region", "variable", "unit"], 
    inplace=True
)

## Step 2: Harmonize Scenario Data to Historical Emissions

In [7]:
harmoniser = create_cmip7_scenariomip_global_harmoniser(
    CMIP7_HISTORICAL_EMISSIONS,
    ANERIS_GLOBAL_OVERRIDES,
    run_checks=False
)

harmonised_data = harmoniser(scenario_data_renamed)

0it [00:00, ?it/s]

Scenarios to harmonise:   0%|          | 0/1522 [00:00<?, ?it/s]

INFO:root:No data in the model dataframe for ('World', 'Emissions', 'Mt BC/yr'), dH_abs_thresh not calculated
INFO:root:No data in the model dataframe for ('World', 'Emissions', 'Mt BC/yr'), dH_abs_thresh not calculated
INFO:root:No data in the model dataframe for ('World', 'Emissions', 'Mt CH4/yr'), dH_abs_thresh not calculated
INFO:root:No data in the model dataframe for ('World', 'Emissions', 'Mt CH4/yr'), dH_abs_thresh not calculated
INFO:root:No data in the model dataframe for ('World', 'Emissions', 'Mt CO/yr'), dH_abs_thresh not calculated
INFO:root:No data in the model dataframe for ('World', 'Emissions|CO2', 'Mt CO2/yr'), dH_abs_thresh not calculated
INFO:root:No data in the model dataframe for ('World', 'Emissions|CO2', 'Mt CO2/yr'), dH_abs_thresh not calculated
INFO:root:No data in the model dataframe for ('World', 'Emissions', 'kt HFC125/yr'), dH_abs_thresh not calculated
INFO:root:No data in the model dataframe for ('World', 'Emissions', 'kt HFC134a/yr'), dH_abs_thresh not 

## Step 3: Prepare for Infilling - Create Intermediate Database

In [8]:
harmonised_pyam = pyam.IamDataFrame(harmonised_data)

# Add total CO2 emissions (Fossil + Biosphere)
harmonised_pyam.add(
    a='Emissions|CO2|Fossil',
    b='Emissions|CO2|Biosphere',
    name='Emissions|CO2',
    ignore_units='Mt CO2/yr',
    append=True
)

# Save intermediate database for infilling
harmonised_pyam.timeseries().to_csv(INTERMEDIATE_OUTPUT)

## Step 4: Infill Missing Variables Using AR6-Style Approach

In [9]:
def infill_missing_variable(variable, harmonised_pyam_data, cruncher_class, lead_vars):
    """Infill a single missing variable using specified database cruncher."""
    df_copy = deepcopy(harmonised_pyam_data)
    df_copy.require_data(variable=variable, exclude_on_fail=True)
    data_to_infill = df_copy.filter(exclude=True)
    
    if data_to_infill.empty:
        return None
    
    print(f"Infilling {variable}")
    cruncher = cruncher_class(harmonised_pyam_data)
    filler = cruncher.derive_relationship(variable, lead_vars)
    return filler(data_to_infill)


# Identify missing variables from CMIP7 infiller
cmip7_infiller = CMIP7ScenarioMIPInfiller.from_cmip7_scenariomip_config(
    cmip7_scenariomip_infilling_leader_emissions_file=CMIP7_INFILLING_FILE,
    cmip7_ghg_inversions_file=GHG_INVERSION_FILE,
    cmip7_scenariomip_global_historical_emissions_file=CMIP7_HISTORICAL_EMISSIONS,
)

available_vars = set(cmip7_infiller.infilling_db.index.get_level_values('variable').unique())
ghg_inversion_vars = set(cmip7_infiller.cmip7_ghg_inversions.index.get_level_values('variable'))
cmip7_vars = available_vars.union(ghg_inversion_vars)

from gcages.cmip7_scenariomip.infilling import COMPLETE_EMISSIONS_INPUT_VARIABLES_GCAGES
missing_variables = list(set(COMPLETE_EMISSIONS_INPUT_VARIABLES_GCAGES).difference(cmip7_vars))

In [10]:
missing_variables

['Emissions|NH3',
 'Emissions|BC',
 'Emissions|OC',
 'Emissions|CO2|Biosphere',
 'Emissions|CO',
 'Emissions|SOx',
 'Emissions|NOx',
 'Emissions|NMVOC',
 'Emissions|CH4']

In [11]:
# Infill all missing variables
infilled_dataframes = []
for variable in missing_variables:
    if variable not in VARS_DB_CRUNCHERS:
        continue
    
    _, cruncher_class = VARS_DB_CRUNCHERS[variable]
    
    try:
        infilled = infill_missing_variable(
            variable, 
            harmonised_pyam, 
            cruncher_class, 
            LEAD_VARIABLES
        )
        if infilled is not None:
            infilled_dataframes.append(infilled)
    except Exception as e:
        print(f"Failed to infill {variable}: {e}")

[INFO] 13:53:09 - pyam.validation: 357 scenarios failed validation and will be set as `exclude=True`.


Infilling Emissions|NH3


/Users/hoegner/GitHub/ar7_netzero_assessment/.venv/lib/python3.13/site-packages/silicone/database_crunchers/quantile_rolling_windows.py:182: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  wide_db = wide_db.applymap(lambda x: np.nan if isinstance(x, str) else x)
[INFO] 13:53:10 - pyam.validation: 364 scenarios failed validation and will be set as `exclude=True`.


Infilling Emissions|BC


/Users/hoegner/GitHub/ar7_netzero_assessment/.venv/lib/python3.13/site-packages/silicone/database_crunchers/quantile_rolling_windows.py:182: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  wide_db = wide_db.applymap(lambda x: np.nan if isinstance(x, str) else x)
[INFO] 13:53:11 - pyam.validation: 347 scenarios failed validation and will be set as `exclude=True`.


...

/Users/hoegner/GitHub/ar7_netzero_assessment/.venv/lib/python3.13/site-packages/silicone/database_crunchers/quantile_rolling_windows.py:182: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  wide_db = wide_db.applymap(lambda x: np.nan if isinstance(x, str) else x)
[INFO] 13:53:16 - pyam.validation: 357 scenarios failed validation and will be set as `exclude=True`.


Infilling Emissions|NMVOC


/Users/hoegner/GitHub/ar7_netzero_assessment/.venv/lib/python3.13/site-packages/silicone/database_crunchers/quantile_rolling_windows.py:182: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  wide_db = wide_db.applymap(lambda x: np.nan if isinstance(x, str) else x)
[INFO] 13:53:17 - pyam.validation: 6 scenarios failed validation and will be set as `exclude=True`.


Infilling Emissions|CH4


/Users/hoegner/GitHub/ar7_netzero_assessment/.venv/lib/python3.13/site-packages/silicone/database_crunchers/quantile_rolling_windows.py:182: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  wide_db = wide_db.applymap(lambda x: np.nan if isinstance(x, str) else x)


## Step 5: Combine Infilled Data with Harmonised Data

In [12]:
intermediate_infilled = pyam.concat(infilled_dataframes)
intermediate_infilled = pyam.concat([intermediate_infilled, harmonised_pyam])

# Remove total CO2 (will be recalculated by CMIP7 infiller)
intermediate_infilled.filter(variable='Emissions|CO2', keep=False, inplace=True)

# Remove problematic scenario
intermediate_infilled.filter(
    model='COFFEE 1.5',
    scenario='NAVIGATE*',
    keep=False,
    inplace=True
)

intermediate_timeseries = intermediate_infilled.timeseries()

## Step 6: Complete Infilling with CMIP7 ScenarioMIP Infiller

In [ ]:
complete_data = cmip7_infiller(intermediate_timeseries)
complete_data

In [ ]:
complete_data.to_csv(OUTPUT_FOLDER / "101_harmonised_data.csv")